<a href="https://colab.research.google.com/github/thisishasan/cbir_system/blob/main/18_extract_feature_subclass_400.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [40]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [41]:
%cd '/content/drive/My Drive/01_information_retrieval'
!ls

/content/drive/My Drive/01_information_retrieval
binary_scenario		      training_1_40
checkpoint		      training_1_400
Notes.gdoc		      training_1_400.h5
original		      training_1_400.json
reconstruction		      training_1_400_subclass
subclass_scenario	      training_1_400_subclass.h5
tflite_model		      training_1_400_subclass.json
training_1_100		      training_1_40.h5
training_1_100.h5	      training_1_40.json
training_1_100.json	      training_1_40_subclass
training_1_100_subclass       training_1_40_subclass.h5
training_1_100_subclass.h5    training_1_40_subclass.json
training_1_100_subclass.json  training_1_indexed_100.json
training_1_200		      training_1_indexed_100_subclass.json
training_1_200.h5	      training_1_indexed_200.json
training_1_200.json	      training_1_indexed_200_subclass.json
training_1_200_subclass       training_1_indexed_400.json
training_1_200_subclass.h5    training_1_indexed_40.json
training_1_200_subclass.json  training_1_indexed_40_subclass.json


In [42]:
import numpy as np
import json
import os
import cv2
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.layers import Conv2D
from tensorflow.keras.layers import Conv2DTranspose
from tensorflow.keras.layers import LeakyReLU
from tensorflow.keras.layers import Activation
from tensorflow.keras.layers import Flatten
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Reshape
from tensorflow.keras.layers import Input
from tensorflow.keras.models import Model
from tensorflow.keras import backend as K

In [43]:
class ConvAutoEncoder:
    @staticmethod
    def build(width, height, depth, filters=(128,), latent_dim=48):
        input_shape = (height, width, depth)
        channel_dim = -1
        inputs = Input(shape=input_shape)
        x = inputs
        # Encoder layer
        for f in filters:
            x = Conv2D(f, (3, 3), strides=2, padding='same')(x)
            x = LeakyReLU(negative_slope=0.2)(x)
            x = BatchNormalization(axis=channel_dim, name='enc_filter_'+str(f))(x)

        volume_size = K.int_shape(x)
        x = Flatten()(x)
        # Latent layer
        latent = Dense(latent_dim, name="encoded")(x)

        # Decoder layer
        x = Dense(int(np.prod(volume_size[1:])))(latent)
        x = Reshape((volume_size[1], volume_size[2], volume_size[3]))(x)

        # Reverse on decoder
        for f in filters[::-1]:
            x = Conv2DTranspose(f, (3, 3), strides=2, padding='same')(x)
            x = LeakyReLU(negative_slope=0.2)(x)
            x = BatchNormalization(axis=channel_dim, name='dec_filter_'+str(f))(x)

        x = Conv2DTranspose(depth, (3, 3), padding="same")(x)
        outputs = Activation("sigmoid", name="decoded")(x)

        auto_encoder = Model(inputs, outputs, name="auto_encoder")
        return auto_encoder

In [44]:
base_dataset = "subclass_scenario"
class_dir = ['tubular_adenoma', 'phyllodes_tumor', 'papillary_carcinoma',
             'mucinous_carcinoma', 'lobular_carcinoma', 'fibroadenoma',
             'ductal_carcinoma', 'adenosis'
             ]
magnification = '400X'
IMAGE_SIZE = (256, 256)

In [45]:
print("[INFO] indexing file images BreaKHis dataset...")
# indexing file images
dataset_train = []
for class_item in class_dir:
    cur_dir = os.path.join(base_dataset, 'train', magnification, class_item)
    for file in os.listdir(cur_dir):
        dataset_train.append(os.path.join(cur_dir, file))

[INFO] indexing file images BreaKHis dataset...


In [46]:
print("train:", len(dataset_train))

train: 1482


In [47]:
print("[INFO] load images BreaKHis dataset...")
#  load images
train_images = []
for image_path in dataset_train:
    if ".png" in image_path:
        image = cv2.imread(image_path)
        image = cv2.resize(image, IMAGE_SIZE)
        train_images.append(image)

[INFO] load images BreaKHis dataset...


In [48]:
# normalization
print("[INFO] normalization...")
train_x = np.array(train_images).astype("float32") / 255.0

[INFO] normalization...


In [49]:
auto_encoder = ConvAutoEncoder.build(IMAGE_SIZE[0], IMAGE_SIZE[1], 3)
# load our auto_encoder from disk
print("[INFO] loading auto encoder model...")
auto_encoder.load_weights("training_1_400_subclass/cp.ckpt.weights.h5")

[INFO] loading auto encoder model...


In [50]:
# create the encoder model which consists of *just* the encoder
# portion of the auto encoder
encoder = Model(inputs=auto_encoder.input,
	outputs=auto_encoder.get_layer("encoded").output)

# quantify the contents of our input images using the encoder
print("[INFO] encoding images...")
features = encoder.predict(train_x)

[INFO] encoding images...
47/47 ━━━━━━━━━━━━━━━━━━━━ 23s 491ms/step


In [51]:
indexes = list(range(0, train_x.shape[0]))
features_array = [[float(x) for x in y] for y in features]
labels = [path.split("/")[3] for path in dataset_train]
data = {"indexes": indexes, "features": features_array, "locations": dataset_train, "labels":labels}

In [52]:
with open('training_1_indexed_400_subclass.json', 'w') as f:
    json.dump(data, f)